In [1]:
# ============================================================
import pandas as pd
import numpy as np
import logging
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 导入自定义模块
from src.core.database import DatabaseManager
from src.core.data_fetcher import DataFetcher
from src.core.spread_calculator import SpreadCalculator, create_standard_crack_spreads
from src.core.indicators import IndicatorBuilder
from src.core.feature_engineering import FeatureEngineer
from src.core.ml_models import MLModel, SignalGenerator
from src.core.backtest import BacktestEngine, PerformanceAnalyzer
from src.core.visualization import Visualizer

# ============================================================
# 配置日志
# ============================================================
log_dir = Path('logs')
log_dir.mkdir(exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_dir / 'debug_strategy.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

logger.info("="*60)
logger.info("开始运行调试版本策略")
logger.info("="*60)

# ============================================================
# 全局变量初始化
# ============================================================
print("\n[1/9] 初始化模块...")

# 数据库和工具类
db_path = "data/trading_data.db"
db = DatabaseManager(db_path)
fetcher = DataFetcher()
spread_calc = SpreadCalculator(db)
indicator_builder = IndicatorBuilder(db)
feature_engineer = FeatureEngineer(indicator_builder)
visualizer = Visualizer()

# 数据存储容器
price_data = {}          # 存储各品种价格数据
spread_data = {}         # 存储价差数据
macro_data = {}          # 存储宏观数据
fundamental_data = {}    # 存储基本面数据

# 特征和模型
features_df = None       # 合并后的特征DataFrame
X_train = None          # 训练集特征
X_test = None           # 测试集特征
y_train = None          # 训练集标签
y_test = None           # 测试集标签
train_idx = None        # 训练集索引
test_idx = None         # 测试集索引
model = None            # 训练好的模型
selected_features = []  # 选择的特征列表

# 回测结果
signals = None          # 交易信号
equity_curve = None     # 权益曲线
trade_log = None        # 交易日志
performance_report = None  # 绩效报告

print("✓ 模块初始化完成")

START_DATE = "2000-01-01"

INFO:__main__:============================================================
INFO:__main__:开始运行调试版本策略
INFO:__main__:============================================================
INFO:src.core.database:数据表创建完成
INFO:src.core.database:数据库已初始化: data/trading_data.db
INFO:__main__:开始运行调试版本策略
INFO:__main__:============================================================
INFO:src.core.database:数据表创建完成
INFO:src.core.database:数据库已初始化: data/trading_data.db



[1/9] 初始化模块...
✓ 模块初始化完成


In [2]:
# ============================================================
# 步骤1：数据获取
# ============================================================

print("\n[2/9] 开始数据获取...")
logger.info("\n" + "="*60)
logger.info("步骤1：数据获取")
logger.info("="*60)

# 期货品种配置
symbols_config = {
    'CL': 'CL=F',    # WTI原油
    'RBOB': 'RB=F',  # RBOB汽油
    'HO': 'HO=F'     # 取暖油/柴油
}

# 获取期货数据
for name, symbol in symbols_config.items():
    print(f"  获取 {name} ({symbol}) 数据...")
    logger.info(f"获取 {name} 数据...")
    
    df = fetcher.fetch_yfinance_data(
        symbol, 
        start_date=START_DATE,
        interval='1d'
    )
    
    if not df.empty:
        # 期货复权处理
        # df_adjusted = FuturesAdjuster.adjust_futures_roll(df, method='ratio')
        df_adjusted = df.copy()  # 简化处理，直接使用原始数据
        # 保存到数据库
        db.insert_price_data(df_adjusted, name, 'adjusted')
        price_data[name] = df_adjusted
        
        print(f"  ✓ {name}: {len(df_adjusted)} 条记录")
        logger.info(f"{name} 数据获取成功: {len(df_adjusted)} 条记录")
    else:
        print(f"  ✗ {name} 数据获取失败")
        logger.warning(f"{name} 数据获取失败")

INFO:__main__:
INFO:__main__:步骤1：数据获取
INFO:__main__:============================================================
INFO:__main__:获取 CL 数据...
INFO:__main__:步骤1：数据获取
INFO:__main__:============================================================
INFO:__main__:获取 CL 数据...



[2/9] 开始数据获取...
  获取 CL (CL=F) 数据...


INFO:src.core.data_fetcher:正在从yfinance获取 CL=F 的数据...
INFO:src.core.data_fetcher:成功获取 6326 条 CL=F 的数据
INFO:src.core.data_fetcher:成功获取 6326 条 CL=F 的数据
INFO:src.core.database:插入了 0 条新数据，跳过了 6326 条重复数据
INFO:__main__:CL 数据获取成功: 6326 条记录
INFO:__main__:获取 RBOB 数据...
INFO:src.core.data_fetcher:正在从yfinance获取 RB=F 的数据...
INFO:src.core.database:插入了 0 条新数据，跳过了 6326 条重复数据
INFO:__main__:CL 数据获取成功: 6326 条记录
INFO:__main__:获取 RBOB 数据...
INFO:src.core.data_fetcher:正在从yfinance获取 RB=F 的数据...
INFO:src.core.data_fetcher:成功获取 6281 条 RB=F 的数据
INFO:src.core.data_fetcher:成功获取 6281 条 RB=F 的数据


  ✓ CL: 6326 条记录
  获取 RBOB (RB=F) 数据...


INFO:src.core.database:插入了 0 条新数据，跳过了 6281 条重复数据
INFO:__main__:RBOB 数据获取成功: 6281 条记录
INFO:__main__:获取 HO 数据...
INFO:src.core.data_fetcher:正在从yfinance获取 HO=F 的数据...
INFO:__main__:RBOB 数据获取成功: 6281 条记录
INFO:__main__:获取 HO 数据...
INFO:src.core.data_fetcher:正在从yfinance获取 HO=F 的数据...
INFO:src.core.data_fetcher:成功获取 6320 条 HO=F 的数据
INFO:src.core.data_fetcher:成功获取 6320 条 HO=F 的数据


  ✓ RBOB: 6281 条记录
  获取 HO (HO=F) 数据...


INFO:src.core.database:插入了 0 条新数据，跳过了 6320 条重复数据
INFO:__main__:HO 数据获取成功: 6320 条记录
INFO:__main__:HO 数据获取成功: 6320 条记录


  ✓ HO: 6320 条记录


In [ ]:
# 获取宏观数据
print("\n  获取宏观经济数据...")
logger.info("获取宏观经济数据...")

# VIX波动率指数
vix_data = fetcher.fetch_index_data('VIX', start_date=START_DATE)
if not vix_data.empty:
    db.insert_price_data(vix_data, 'VIX', 'index')
    macro_data['VIX'] = vix_data
    print(f"  ✓ VIX: {len(vix_data)} 条记录")
    logger.info(f"VIX数据获取成功: {len(vix_data)} 条记录")

# 美元指数
dxy_data = fetcher.fetch_index_data('DXY', start_date=START_DATE)
if not dxy_data.empty:
    db.insert_price_data(dxy_data, 'DXY', 'index')
    macro_data['DXY'] = dxy_data
    print(f"  ✓ DXY: {len(dxy_data)} 条记录")
    logger.info(f"DXY数据获取成功: {len(dxy_data)} 条记录")


# 获取基本面数据
print("\n  获取基本面数据...")
logger.info("获取基本面数据...")
eia_data = fetcher.fetch_eia_data('PET.WCRSTUS1.W')
if not eia_data.empty:
    for date, row in eia_data.iterrows():
        db.insert_fundamental_data(
            'EIA',
            date.strftime('%Y-%m-%d'),
            date.strftime('%Y-%m-%d'),
            row.to_dict()
        )
    fundamental_data['EIA'] = eia_data
    print(f"  ✓ EIA: {len(eia_data)} 条记录")
    logger.info(f"EIA数据获取成功: {len(eia_data)} 条记录")


print("✓ 数据获取完成")
logger.info("数据获取完成\n")

# 🔍 调试点1：在此处设置断点，检查 price_data, macro_data 的内容
# 可以在调试控制台输入: price_data.keys(), len(price_data['CL'])

INFO:__main__:获取宏观经济数据...
INFO:src.core.data_fetcher:正在从yfinance获取 ^VIX 的数据...
INFO:src.core.data_fetcher:正在从yfinance获取 ^VIX 的数据...
INFO:src.core.data_fetcher:成功获取 6498 条 ^VIX 的数据
INFO:src.core.data_fetcher:成功获取 6498 条 ^VIX 的数据



  获取宏观经济数据...


INFO:src.core.database:插入了 0 条新数据，跳过了 6498 条重复数据
INFO:__main__:VIX数据获取成功: 6498 条记录
INFO:src.core.data_fetcher:正在从yfinance获取 DX-Y.NYB 的数据...
INFO:__main__:VIX数据获取成功: 6498 条记录
INFO:src.core.data_fetcher:正在从yfinance获取 DX-Y.NYB 的数据...
INFO:src.core.data_fetcher:成功获取 6527 条 DX-Y.NYB 的数据
INFO:src.core.data_fetcher:成功获取 6527 条 DX-Y.NYB 的数据


  ✓ VIX: 6498 条记录


INFO:src.core.database:插入了 0 条新数据，跳过了 6527 条重复数据
INFO:__main__:DXY数据获取成功: 6527 条记录
INFO:__main__:获取基本面数据...
INFO:__main__:DXY数据获取成功: 6527 条记录
INFO:__main__:获取基本面数据...


  ✓ DXY: 6527 条记录

  获取基本面数据...


INFO:__main__:EIA数据获取成功: 209 条记录
INFO:__main__:数据获取完成

INFO:__main__:EIA数据获取成功: 209 条记录
INFO:__main__:数据获取完成



  ✓ EIA: 209 条记录
✓ 数据获取完成


In [4]:
# ============================================================
# 步骤2：计算价差
# ============================================================
print("\n[3/9] 开始计算价差...")
logger.info("\n" + "="*60)
logger.info("步骤2：计算价差")
logger.info("="*60)


# 创建标准裂解价差配置

print(price_data["CL"].head(20))
create_standard_crack_spreads(spread_calc)

# 计算3:2:1裂解价差
if all(symbol in price_data for symbol in ['CL', 'RBOB', 'HO']):
    print("  计算 CRACK_3_2_1 价差...")
    
    spread_df = spread_calc.calculate_spread(
        'CRACK_3_2_1',
        price_data,
        price_column='close'
    )
    
    # 添加统计特征
    spread_df = spread_calc.get_spread_statistics(spread_df, window=20)
    spread_data['CRACK_3_2_1'] = spread_df
    
    # 保存到数据库
    db.insert_indicator_data(
        'CRACK_3_2_1',
        spread_df[['spread']],
        metadata={'type': 'crack_spread', 'ratio': '3:2:1'}
    )
    
    print(f"  ✓ 价差数据点: {len(spread_df)}")
    logger.info(f"3:2:1裂解价差计算完成: {len(spread_df)} 个数据点")
    
    # 平稳性检验
    print("  进行ADF平稳性检验...")
    indicator_builder.test_stationarity(
        spread_df['spread'],
        name='CRACK_3_2_1 Spread'
    )
else:
    print("  ✗ 缺少必要的价格数据")
    logger.error("缺少计算价差所需的价格数据")

print("✓ 价差计算完成")
logger.info("价差计算完成\n")

# 🔍 调试点2：在此处设置断点，检查 spread_data['CRACK_3_2_1'] 的统计特征
print(spread_data['CRACK_3_2_1'].head(20))

INFO:__main__:
INFO:__main__:步骤2：计算价差
INFO:__main__:============================================================
INFO:__main__:步骤2：计算价差
INFO:__main__:============================================================
INFO:src.core.spread_calculator:价差配置已保存到数据库: RBOB_CL_1_1
INFO:src.core.spread_calculator:创建价差配置: RBOB_CL_1_1 - 多头(1.0xRBOB) - 空头(1.0xCL)
INFO:src.core.spread_calculator:价差配置已保存到数据库: HO_CL_1_1
INFO:src.core.spread_calculator:创建价差配置: HO_CL_1_1 - 多头(1.0xHO) - 空头(1.0xCL)
INFO:src.core.spread_calculator:价差配置已保存到数据库: CRACK_3_2_1
INFO:src.core.spread_calculator:价差配置已保存到数据库: RBOB_CL_1_1
INFO:src.core.spread_calculator:创建价差配置: RBOB_CL_1_1 - 多头(1.0xRBOB) - 空头(1.0xCL)
INFO:src.core.spread_calculator:价差配置已保存到数据库: HO_CL_1_1
INFO:src.core.spread_calculator:创建价差配置: HO_CL_1_1 - 多头(1.0xHO) - 空头(1.0xCL)
INFO:src.core.spread_calculator:价差配置已保存到数据库: CRACK_3_2_1
INFO:src.core.spread_calculator:创建价差配置: CRACK_3_2_1 - 多头(2.0xRBOB + 1.0xHO) - 空头(3.0xCL)
INFO:src.core.spread_calculator:价差配置已保存到数据库: CRACK


[3/9] 开始计算价差...
                                open       high        low      close  volume  \
Date                                                                            
2000-08-23 00:00:00-04:00  31.950001  32.799999  31.950001  32.049999   79385   
2000-08-24 00:00:00-04:00  31.900000  32.240002  31.400000  31.629999   72978   
2000-08-25 00:00:00-04:00  31.700001  32.099998  31.320000  32.049999   44601   
2000-08-28 00:00:00-04:00  32.040001  32.919998  31.860001  32.869999   46770   
2000-08-29 00:00:00-04:00  32.820000  33.029999  32.560001  32.720001   49131   
2000-08-30 00:00:00-04:00  32.750000  33.400002  32.099998  33.400002   79214   
2000-08-31 00:00:00-04:00  33.250000  33.700001  32.970001  33.099998   56895   
2000-09-01 00:00:00-04:00  33.049999  33.450001  32.750000  33.380001   45869   
2000-09-05 00:00:00-04:00  33.950001  33.990002  33.419998  33.799999   55722   
2000-09-06 00:00:00-04:00  33.990002  34.950001  33.830002  34.950001   74692   
2000-09-07 

INFO:src.core.spread_calculator:价差统计特征计算完成，窗口: 20
INFO:src.core.database:插入了 0 条新指标数据
INFO:__main__:3:2:1裂解价差计算完成: 6276 个数据点
INFO:src.core.database:插入了 0 条新指标数据
INFO:__main__:3:2:1裂解价差计算完成: 6276 个数据点


  ✓ 价差数据点: 6276
  进行ADF平稳性检验...


INFO:src.core.indicators:
INFO:src.core.indicators:ADF平稳性检验结果 - CRACK_3_2_1 Spread
INFO:src.core.indicators:==================================================
INFO:src.core.indicators:ADF统计量: -3.749940
INFO:src.core.indicators:P值: 0.003463
INFO:src.core.indicators:使用滞后阶数: 8
INFO:src.core.indicators:观测值数量: 6267
INFO:src.core.indicators:临界值:
INFO:src.core.indicators:  1%: -3.431394
INFO:src.core.indicators:  5%: -2.862001
INFO:src.core.indicators:  10%: -2.567016
INFO:src.core.indicators:结论: CRACK_3_2_1 Spread 是平稳序列 (p < 0.05)
INFO:src.core.indicators:==================================================

INFO:src.core.indicators:ADF平稳性检验结果 - CRACK_3_2_1 Spread
INFO:src.core.indicators:==================================================
INFO:src.core.indicators:ADF统计量: -3.749940
INFO:src.core.indicators:P值: 0.003463
INFO:src.core.indicators:使用滞后阶数: 8
INFO:src.core.indicators:观测值数量: 6267
INFO:src.core.indicators:临界值:
INFO:src.core.indicators:  1%: -3.431394
INFO:src.core.indicators:  5%: -2.8

✓ 价差计算完成
                              spread  spread_name  spread_mean  spread_std  \
Date                                                                         
2000-11-01 00:00:00-05:00  14.173799  CRACK_3_2_1          NaN         NaN   
2000-11-02 00:00:00-05:00  14.494796  CRACK_3_2_1          NaN         NaN   
2000-11-03 00:00:00-05:00  13.949403  CRACK_3_2_1          NaN         NaN   
2000-11-06 00:00:00-05:00  14.059797  CRACK_3_2_1          NaN         NaN   
2000-11-07 00:00:00-05:00  14.514593  CRACK_3_2_1          NaN         NaN   
2000-11-08 00:00:00-05:00  13.853395  CRACK_3_2_1          NaN         NaN   
2000-11-09 00:00:00-05:00  14.624399  CRACK_3_2_1          NaN         NaN   
2000-11-10 00:00:00-05:00  13.293000  CRACK_3_2_1          NaN         NaN   
2000-11-13 00:00:00-05:00  13.734597  CRACK_3_2_1          NaN         NaN   
2000-11-14 00:00:00-05:00  14.675407  CRACK_3_2_1          NaN         NaN   
2000-11-15 00:00:00-05:00  15.970204  CRACK_3_2_1      

### 📊 价格单位转换说明

在计算裂解价差时，不同产品的价格单位不同：
- **WTI原油 (CL)**: 美元/桶
- **RBOB汽油 (RBOB)**: 美元/加仑
- **取暖油/柴油 (HO)**: 美元/加仑

为了正确计算价差，需要将所有价格**统一转换为美元/桶**：
- 1桶 = 42加仑
- RBOB和HO的价格需要乘以42

**转换示例**：
```python
# RBOB汽油: 2.50 美元/加仑 → 105.00 美元/桶 (2.50 × 42)
# WTI原油:  70.00 美元/桶 → 70.00 美元/桶 (不变)
```

`SpreadCalculator.calculate_spread()` 方法会自动进行单位转换。

In [7]:
# ============================================================
# 价格单位验证和转换演示
# ============================================================

print("\n[补充] 价格单位转换验证...")
print("="*60)

# 显示原始价格（最近5天）
print("\n📊 原始价格（最近5天）:")
print("-"*60)

for symbol in ['CL', 'RBOB', 'HO']:
    if symbol in price_data:
        recent_prices = price_data[symbol]['close'].tail(5)
        avg_price = recent_prices.mean()
        
        unit = "美元/桶" if symbol == 'CL' else "美元/加仑"
        print(f"\n{symbol} ({unit}):")
        print(f"  平均价格: {avg_price:.4f} {unit}")
        print(f"  价格范围: {recent_prices.min():.4f} - {recent_prices.max():.4f}")

# 演示单位转换
print("\n\n🔄 单位转换演示:")
print("-"*60)

from src.core.spread_calculator import GALLONS_PER_BARREL

for symbol in ['RBOB', 'HO']:
    if symbol in price_data:
        original_price = price_data[symbol]['close'].tail(5).mean()
        converted_price = original_price * GALLONS_PER_BARREL
        
        print(f"\n{symbol}:")
        print(f"  原始价格:   {original_price:.4f} 美元/加仑")
        print(f"  转换系数:   × {GALLONS_PER_BARREL} (加仑/桶)")
        print(f"  转换后价格: {converted_price:.4f} 美元/桶")

# 验证价差计算是否使用了单位转换
print("\n\n✅ 价差计算验证:")
print("-"*60)

if 'CRACK_3_2_1' in spread_data:
    spread_df = spread_data['CRACK_3_2_1']
    avg_spread = spread_df['spread'].tail(20).mean()
    std_spread = spread_df['spread'].tail(20).std()
    
    print(f"\nCRACK_3_2_1 价差统计（最近20天）:")
    print(f"  平均值: {avg_spread:.4f} 美元/桶")
    print(f"  标准差: {std_spread:.4f} 美元/桶")
    print(f"  范围:   {spread_df['spread'].tail(20).min():.4f} - {spread_df['spread'].tail(20).max():.4f}")
    
    # 检查价差是否在合理范围（转换后）
    if abs(avg_spread) > 200:
        print("\n⚠️ 警告: 价差值异常大，可能未进行单位转换！")
    else:
        print("\n✅ 价差值在合理范围内，单位转换正常")

print("\n" + "="*60)


[补充] 价格单位转换验证...

📊 原始价格（最近5天）:
------------------------------------------------------------

CL (美元/桶):
  平均价格: 60.6980 美元/桶
  价格范围: 60.1500 - 61.3100

RBOB (美元/加仑):
  平均价格: 1.9632 美元/加仑
  价格范围: 1.9204 - 2.0034

HO (美元/加仑):
  平均价格: 2.4278 美元/加仑
  价格范围: 2.3872 - 2.4600


🔄 单位转换演示:
------------------------------------------------------------

RBOB:
  原始价格:   1.9632 美元/加仑
  转换系数:   × 42 (加仑/桶)
  转换后价格: 82.4527 美元/桶

HO:
  原始价格:   2.4278 美元/加仑
  转换系数:   × 42 (加仑/桶)
  转换后价格: 101.9659 美元/桶


✅ 价差计算验证:
------------------------------------------------------------

CRACK_3_2_1 价差统计（最近20天）:
  平均值: 75.0320 美元/桶
  标准差: 6.6373 美元/桶
  范围:   68.7984 - 89.8956

✅ 价差值在合理范围内，单位转换正常



In [8]:
# ============================================================
# 步骤3：特征工程
# ============================================================
print("\n[4/9] 开始特征工程...")
logger.info("\n" + "="*60)
logger.info("步骤3：特征工程")
logger.info("="*60)

# 获取主价差数据
main_spread = spread_data.get('CRACK_3_2_1')
if main_spread is None or main_spread.empty:
    print("  ✗ 价差数据不可用，停止执行")
    logger.error("价差数据不可用")
    raise ValueError("价差数据不可用")

# 1. 创建价差特征
print("  [3.1] 创建价差特征...")
spread_features = feature_engineer.create_spread_features(main_spread)
print(f"    ✓ 价差特征: {len(spread_features.columns)} 个")

# 2. 创建价格特征
print("  [3.2] 创建价格特征...")
price_features = feature_engineer.create_price_features(price_data)
print(f"    ✓ 价格特征: {len(price_features.columns)} 个")

# 3. 创建技术指标特征
print("  [3.3] 创建技术指标特征...")
technical_features = pd.DataFrame(index=main_spread.index)
for symbol, df in price_data.items():
    if symbol in ['CL', 'RBOB', 'HO']:
        tech_df = feature_engineer.create_technical_features(df, symbol)
        # 选择关键列
        key_cols = [col for col in tech_df.columns 
                   if any(x in col for x in ['RSI', 'MACD', 'BB_percent'])]
        if key_cols:
            technical_features = technical_features.join(tech_df[key_cols], how='outer')
print(f"    ✓ 技术指标特征: {len(technical_features.columns)} 个")

# 4. 创建季节性特征
print("  [3.4] 创建季节性特征...")
seasonal_features = feature_engineer.create_seasonal_features(
    pd.DataFrame(index=main_spread.index)
)
print(f"    ✓ 季节性特征: {len(seasonal_features.columns)} 个")

# 5. 创建宏观特征（使用merge_asof对齐时间戳）
print("  [3.5] 创建宏观特征...")
# 创建基准DataFrame - 使用reset_index()确保正确的datetime类型
# 关键：直接赋值 macro_features['_timestamp'] = macro_features.index 会导致类型变为object
temp_macro = main_spread.reset_index()
temp_macro.columns = ['date'] + list(main_spread.columns)

# 确保date列是datetime类型且无时区
temp_macro['date'] = pd.to_datetime(temp_macro['date'])
if hasattr(temp_macro['date'].dtype, 'tz') and temp_macro['date'].dtype.tz is not None:
    temp_macro['date'] = temp_macro['date'].dt.tz_localize(None)

print(f"主数据时间类型: {temp_macro['date'].dtype}, 前5行:")
print(temp_macro[['date']].head())

for symbol in ['VIX', 'DXY']:
    df = db.get_price_data(symbol)
    if not df.empty and 'close' in df.columns:
        # 准备宏观数据：重置索引，计算收益率
        macro_df = df[['close']].copy()
        macro_df[f'{symbol}_return_1d'] = macro_df['close'].pct_change()
        macro_df = macro_df.reset_index()
        macro_df.columns = ['date', f'{symbol}_close', f'{symbol}_return_1d']
        
        # 确保有干净的datetime列（不带时区）
        # 如果原数据带时区，先用utc=True转换，再移除时区
        macro_df['date'] = pd.to_datetime(macro_df['date'], utc=True)
        if hasattr(macro_df['date'].dtype, 'tz') and macro_df['date'].dtype.tz is not None:
            macro_df['date'] = macro_df['date'].dt.tz_localize(None)
        
        print(f"{symbol}数据时间类型: {macro_df['date'].dtype}")
        
        # 使用merge_asof进行时间对齐（向后填充）
        temp_macro = pd.merge_asof(
            temp_macro.sort_values('date'),
            macro_df[['date', f'{symbol}_close', f'{symbol}_return_1d']].sort_values('date'),
            on='date',
            direction='backward'  # 使用最近的历史数据
        )
        
        # 显示对齐效果
        aligned_count = temp_macro[f'{symbol}_close'].notna().sum()
        missing_count = temp_macro[f'{symbol}_close'].isnull().sum()
        print(f"✓ {symbol}: {len(df)}条原始 → {aligned_count}条对齐（缺失{missing_count}个）")

# 恢复为索引格式（确保索引无时区）
macro_features = temp_macro.set_index('date')
# 再次确认索引无时区
if hasattr(macro_features.index.dtype, 'tz') and macro_features.index.tz is not None:
    macro_features.index = macro_features.index.tz_localize(None)

# 只保留宏观数据列，去除来自main_spread的列（避免与spread_features重复）
macro_cols = [col for col in macro_features.columns if any(x in col for x in ['VIX', 'DXY'])]
macro_features = macro_features[macro_cols]

print("\n宏观特征前20行:")
print(macro_features.head(20))
print(f"宏观特征索引类型: {macro_features.index.dtype}")

print(f"    ✓ 宏观特征: {len(macro_features.columns)} 个（已时间对齐）")

# 6. 创建目标变量
print("  [3.6] 创建目标变量...")
target_df = feature_engineer.create_target_variable(
    main_spread,
    method='sharpe_regress',
    forward_period=10,
    threshold=0.05
)
print(f"    ✓ 目标变量创建完成")

# 🔍 调试点3：在此处设置断点，检查各个特征DataFrame
# 可以查看: spread_features.head(), price_features.shape, technical_features.columns

# ============================================================
# 合并特征
# ============================================================
print("\n  [3.7] 合并所有特征...")
logger.info("合并特征...")

all_features_list = [
    spread_features,
    price_features,
    technical_features,
    seasonal_features,
    macro_features,
    target_df[['target']]
]
# print(pd.concat(all_features_list, axis=1).head(20))

# 诊断：打印合并前的状态
print("\n  诊断信息 - 合并前各DataFrame状态:")

df_names = ['价差特征', '价格特征', '技术指标', '季节性特征', '宏观特征', '目标变量']

# 统一清理所有DataFrame的索引时区
print("\n  [3.7.1] 统一清理索引时区...")
for i, (name, df) in enumerate(zip(df_names, all_features_list)):
    # 检查索引是否有时区
    if hasattr(df.index, 'tz') and df.index.tz is not None:
        print(f"    {name}: 移除时区 {df.index.tz}")
        all_features_list[i].index = df.index.tz_localize(None)
    
    # 打印诊断信息
    null_count = df.isnull().sum().sum()
    index_type = type(df.index).__name__
    index_dtype = df.index.dtype if hasattr(df.index, 'dtype') else 'N/A'
    print(f"    {name}: {len(df)}样本, {len(df.columns)}列, {null_count}缺失值, 索引类型:{index_type}({index_dtype})")
    if null_count > 0:
        null_cols = df.isnull().sum()
        null_cols = null_cols[null_cols > 0]
        print(f"      缺失值列: {dict(list(null_cols.items())[:3])}")

# 合并特征
print("\n  [3.7.2] 开始合并特征...")
features_df = feature_engineer.merge_all_features(all_features_list)

print(f"\n  合并后状态:")
print(f"    总样本数: {len(features_df)}")
print(f"    总特征数: {len(features_df.columns)}")
print(f"    总缺失值: {features_df.isnull().sum().sum()}")

# ============================================================
# 清理缺失值
# ============================================================
print("\n  [3.8] 清理缺失值...")

# 显示缺失值最多的列
total_nulls = features_df.isnull().sum().sum()
if total_nulls > 0:
    null_counts = features_df.isnull().sum()
    cols_with_nulls = null_counts[null_counts > 0].sort_values(ascending=False)
    print(f"    缺失值最多的前5列:")
    for col, count in cols_with_nulls.head(5).items():
        pct = count / len(features_df) * 100
        print(f"      {col}: {count} ({pct:.2f}%)")

# 步骤1：前向填充
print("\n    步骤1: ffill前向填充...")
features_df = features_df.fillna(method='ffill')
remaining_nulls = features_df.isnull().sum().sum()
print(f"      剩余缺失值: {remaining_nulls}")

# 步骤2：后向填充
if remaining_nulls > 0:
    print("    步骤2: bfill后向填充...")
    features_df = features_df.fillna(method='bfill')
    remaining_nulls = features_df.isnull().sum().sum()
    print(f"      剩余缺失值: {remaining_nulls}")

# 步骤3：均值填充
if remaining_nulls > 0:
    print("    步骤3: 均值填充...")
    numeric_cols = features_df.select_dtypes(include=[np.number]).columns
    features_df[numeric_cols] = features_df[numeric_cols].fillna(
        features_df[numeric_cols].mean()
    )
    remaining_nulls = features_df.isnull().sum().sum()
    print(f"      剩余缺失值: {remaining_nulls}")

# 步骤4：删除仍有缺失值的列
if remaining_nulls > 0:
    null_cols = features_df.columns[features_df.isnull().any()].tolist()
    print(f"    步骤4: 删除 {len(null_cols)} 个仍有缺失值的列")
    print(f"      删除的列: {null_cols[:5]}")
    features_df = features_df.dropna(axis=1)

print(f"\n  最终清理结果:")
print(f"    剩余样本数: {len(features_df)}")
print(f"    剩余特征数: {len(features_df.columns) - 2}")  # 减去target和forward_return
print(f"    缺失值: {features_df.isnull().sum().sum()}")

print("✓ 特征工程完成")
logger.info(f"特征构建完成，总特征数: {len(features_df.columns) - 2}")
logger.info(f"样本数: {len(features_df)}")

# 🔍 调试点4：在此处设置断点，检查 features_df
# 可以使用: features_df.describe(), features_df.head(), features_df.isnull().sum()

INFO:__main__:
INFO:__main__:步骤3：特征工程
INFO:__main__:============================================================
INFO:__main__:步骤3：特征工程
INFO:__main__:============================================================
INFO:src.core.feature_engineering:创建价差特征完成，特征数: 39
INFO:src.core.feature_engineering:创建价差特征完成，特征数: 39
INFO:src.core.feature_engineering:创建价格特征完成，特征数: 48
INFO:src.core.indicators:计算RSI完成，周期: 14
INFO:src.core.indicators:计算MACD完成，参数: 12/26/9
INFO:src.core.feature_engineering:创建价格特征完成，特征数: 48
INFO:src.core.indicators:计算RSI完成，周期: 14
INFO:src.core.indicators:计算MACD完成，参数: 12/26/9
INFO:src.core.indicators:计算布林带完成，窗口: 20, 标准差: 2.0
INFO:src.core.feature_engineering:创建技术指标特征完成: CL_
INFO:src.core.indicators:计算RSI完成，周期: 14
INFO:src.core.indicators:计算MACD完成，参数: 12/26/9
INFO:src.core.indicators:计算布林带完成，窗口: 20, 标准差: 2.0
INFO:src.core.feature_engineering:创建技术指标特征完成: CL_
INFO:src.core.indicators:计算RSI完成，周期: 14
INFO:src.core.indicators:计算MACD完成，参数: 12/26/9
INFO:src.core.indicators:计算布林带完成，窗口: 20, 


[4/9] 开始特征工程...
  [3.1] 创建价差特征...
    ✓ 价差特征: 51 个
  [3.2] 创建价格特征...
    ✓ 价格特征: 48 个
  [3.3] 创建技术指标特征...
    ✓ 技术指标特征: 15 个
  [3.4] 创建季节性特征...
    ✓ 季节性特征: 5 个
  [3.5] 创建宏观特征...
主数据时间类型: datetime64[ns], 前5行:
        date
0 2000-11-01
1 2000-11-02
2 2000-11-03
3 2000-11-06
4 2000-11-07

VIX数据时间类型: datetime64[ns]
✓ VIX: 6498条原始 → 6276条对齐（缺失0个）
VIX数据时间类型: datetime64[ns]
✓ VIX: 6498条原始 → 6276条对齐（缺失0个）


INFO:src.core.feature_engineering:创建夏普比率回归目标变量
INFO:__main__:合并特征...
INFO:__main__:合并特征...
INFO:src.core.feature_engineering:特征合并完成，总特征数: 124, 样本数: 6276
INFO:src.core.feature_engineering:特征合并完成，总特征数: 124, 样本数: 6276
INFO:__main__:特征构建完成，总特征数: 122
INFO:__main__:样本数: 6276
INFO:__main__:特征构建完成，总特征数: 122
INFO:__main__:样本数: 6276


DXY数据时间类型: datetime64[ns]
✓ DXY: 6527条原始 → 6276条对齐（缺失0个）

宏观特征前20行:
            VIX_close  VIX_return_1d   DXY_close  DXY_return_1d
date                                                           
2000-11-01  23.629999      -0.071877  116.650002      -0.005117
2000-11-02  24.280001       0.027507  115.559998      -0.009344
2000-11-03  23.920000      -0.014827  115.610001       0.000433
2000-11-06  23.670000      -0.010452  114.970001      -0.005536
2000-11-07  24.520000       0.035910  115.690002       0.006263
2000-11-08  24.910000       0.015905  115.519997      -0.001469
2000-11-09  25.660000       0.030108  116.230003       0.006146
2000-11-10  27.200001       0.060016  115.339996      -0.007657
2000-11-13  28.530001       0.048897  115.790001       0.003902
2000-11-14  29.059999       0.018577  115.699997      -0.000777
2000-11-15  26.809999      -0.077426  116.220001       0.004494
2000-11-16  26.150000      -0.024618  116.349998       0.001119
2000-11-17  25.049999      -0.042065

In [25]:
import importlib
import src.core.ml_models
importlib.reload(src.core.ml_models)
from src.core.ml_models import MLModel,SignalGenerator

In [9]:
# ============================================================
# 步骤4：模型训练
# ============================================================
print("\n[5/9] 开始模型训练...")
logger.info("\n" + "="*60)
logger.info("步骤4：模型训练")
logger.info("="*60)

if features_df is None or features_df.empty:
    print("  ✗ 特征数据不可用，停止执行")
    logger.error("特征数据不可用")
    raise ValueError("特征数据不可用")

# 特征选择
print("  [4.1] 特征选择...")
selected_features = feature_engineer.select_features(
    features_df,
    target_col='target',
    method='variance',
    top_k=50
)
print(f"    ✓ 选择了 {len(selected_features)} 个特征")

# 创建模型
print("  [4.2] 创建模型...")
model = MLModel(model_type='gradient_boosting', task='regression')
print("    ✓ 使用 Gradient Boosting 分类器")

# 准备数据
print("  [4.3] 准备训练/测试数据...")
X_train, X_test, y_train, y_test, train_idx, test_idx = model.prepare_data(
    features_df,
    target_col='target',
    feature_cols=selected_features,
    test_size=0.15,
    scale=True
)
print(f"    ✓ 训练集: {len(X_train)} 样本")
print(f"    ✓ 测试集: {len(X_test)} 样本")

# 训练模型
print("  [4.4] 训练模型...")
model_params = {
    'n_estimators': 500,
    'max_depth': 8,
    'learning_rate': 0.1,
    'random_state': 42
}
model.train(X_train, y_train, **model_params)
print("    ✓ 模型训练完成")

# 评估模型
print("  [4.5] 评估模型...")
metrics = model.evaluate(X_test, y_test)
print(f"    ✓ 准确率: {metrics.get('accuracy', 0):.4f}")
print(f"    ✓ F1分数: {metrics.get('f1', 0):.4f}")

# 可视化特征重要性
if model.feature_importance is not None:
    print("  [4.6] 生成特征重要性图...")
    visualizer.plot_feature_importance(model.feature_importance, top_n=15)
    print("    ✓ 特征重要性图已保存")

# 保存模型
print("  [4.7] 保存模型...")
model_path = Path('models/crack_spread_model.pkl')
model_path.parent.mkdir(exist_ok=True)
model.save_model(str(model_path))
print(f"    ✓ 模型已保存到: {model_path}")

print("✓ 模型训练完成")
logger.info("模型训练完成\n")

# 🔍 调试点5：在此处设置断点，检查模型和训练结果
# 可以查看: model.feature_importance, metrics, X_train.shape, X_test.shape

# ============================================================
# 步骤5：回测
# ============================================================
print("\n[6/9] 开始回测...")
logger.info("\n" + "="*60)
logger.info("步骤5：回测")
logger.info("="*60)





INFO:__main__:
INFO:__main__:步骤4：模型训练
INFO:__main__:============================================================
INFO:src.core.feature_engineering:特征选择完成，选择了 50 个特征
INFO:__main__:步骤4：模型训练
INFO:__main__:============================================================
INFO:src.core.feature_engineering:特征选择完成，选择了 50 个特征
INFO:src.core.ml_models:数据准备完成:
INFO:src.core.ml_models:  特征数: 50
INFO:src.core.ml_models:  训练集样本数: 5334
INFO:src.core.ml_models:  测试集样本数: 942
INFO:src.core.ml_models:开始训练 gradient_boosting 模型...
INFO:src.core.ml_models:数据准备完成:
INFO:src.core.ml_models:  特征数: 50
INFO:src.core.ml_models:  训练集样本数: 5334
INFO:src.core.ml_models:  测试集样本数: 942
INFO:src.core.ml_models:开始训练 gradient_boosting 模型...



[5/9] 开始模型训练...
  [4.1] 特征选择...
    ✓ 选择了 50 个特征
  [4.2] 创建模型...
    ✓ 使用 Gradient Boosting 分类器
  [4.3] 准备训练/测试数据...
    ✓ 训练集: 5334 样本
    ✓ 测试集: 942 样本
  [4.4] 训练模型...


INFO:src.core.ml_models:Top 10 重要特征:
INFO:src.core.ml_models:              feature  importance
0     CL_volume_ma_20    0.040501
37      spread_std_60    0.039352
1   RBOB_volume_ma_20    0.038685
24           CL_ma_50    0.038498
2     HO_volume_ma_20    0.037730
32    spread_diff_10d    0.034606
38     spread_kurt_60    0.034148
43              month    0.032486
34          VIX_close    0.031335
22           CL_ma_10    0.029367
INFO:src.core.ml_models:模型训练完成
INFO:src.core.ml_models:              feature  importance
0     CL_volume_ma_20    0.040501
37      spread_std_60    0.039352
1   RBOB_volume_ma_20    0.038685
24           CL_ma_50    0.038498
2     HO_volume_ma_20    0.037730
32    spread_diff_10d    0.034606
38     spread_kurt_60    0.034148
43              month    0.032486
34          VIX_close    0.031335
22           CL_ma_10    0.029367
INFO:src.core.ml_models:模型训练完成
INFO:src.core.ml_models:
模型评估结果:
INFO:src.core.ml_models:MSE: 11.471730
INFO:src.core.ml_models:RMSE: 3.3

    ✓ 模型训练完成
  [4.5] 评估模型...
    ✓ 准确率: 0.0000
    ✓ F1分数: 0.0000
  [4.6] 生成特征重要性图...


INFO:src.core.visualization:特征重要性图表已保存: outputs/charts/feature_importance.png
INFO:src.core.ml_models:模型已保存: models\crack_spread_model.pkl
INFO:__main__:模型训练完成

INFO:__main__:
INFO:__main__:步骤5：回测
INFO:__main__:============================================================
INFO:src.core.ml_models:模型已保存: models\crack_spread_model.pkl
INFO:__main__:模型训练完成

INFO:__main__:
INFO:__main__:步骤5：回测
INFO:__main__:============================================================


    ✓ 特征重要性图已保存
  [4.7] 保存模型...
    ✓ 模型已保存到: models\crack_spread_model.pkl
✓ 模型训练完成

[6/9] 开始回测...


In [10]:
model.predict(X_test)

array([-2.08498543e+00, -6.61210566e-01, -2.15977777e+00, -7.23248112e-01,
       -1.89928843e+00, -2.12663717e+00, -1.82886934e+00, -8.80026683e-01,
       -5.06302622e-01,  1.03260671e+00,  1.09329340e+00,  3.87283080e-01,
        6.98827468e-03,  1.20116586e-01, -2.08626895e+00, -5.92404823e-01,
       -1.08420722e+00, -2.70241779e+00, -2.52793670e+00, -3.54519303e+00,
       -5.33563748e+00, -4.06069396e+00, -5.87202929e+00, -1.36088350e+00,
        8.62153496e-01,  8.57514769e-03, -7.42042594e-01, -6.33758164e-01,
       -1.81818719e+00, -4.07326169e+00, -3.50942925e+00, -1.87054900e+00,
       -2.15515954e+00, -4.53779469e+00, -5.55744117e+00, -4.96754924e+00,
       -2.32081099e+00, -2.91678607e+00, -2.25167783e+00, -1.91675029e+00,
       -1.64623738e+00, -1.45444384e+00, -1.63464007e+00, -1.61786207e+00,
       -2.34716539e+00, -2.87054696e+00, -3.20434457e+00, -3.29376553e+00,
       -3.20234658e+00, -4.02063707e+00, -5.05381888e+00, -4.79596138e+00,
       -5.17233176e+00, -

In [14]:
# 生成信号
print("  [5.1] 生成交易信号...")
signal_generator = SignalGenerator(model, threshold=0.05, 
                                use_rolling_quantile=True,      # ✅ 使用滚动分位数
                                rolling_window=120,              # 120天窗口
                                upper_quantile=0.8,            # 80%分位数
                                lower_quantile=0.2,            # 20%分位数
                                signal_holding_days=3        # 信号维持20天
                                   )
signals = signal_generator.generate_signals(X_test, use_probability=True)
signals.index = test_idx
print(f"    ✓ 生成 {len(signals)} 个信号")
print(f"    信号分布: {signals.value_counts().to_dict()}")

# 获取价差价格数据
print("  [5.2] 准备价格数据...")
# 确保spread_data索引与test_idx时区一致
spread_df_for_backtest = spread_data['CRACK_3_2_1'].copy()
if hasattr(spread_df_for_backtest.index, 'tz') and spread_df_for_backtest.index.tz is not None:
    spread_df_for_backtest.index = spread_df_for_backtest.index.tz_localize(None)

spread_prices = spread_df_for_backtest.loc[test_idx, ['spread']].copy()
spread_prices.columns = ['close']
spread_prices['volatility'] = spread_prices['close'].pct_change().rolling(20).std()
print(f"    ✓ 价格数据: {len(spread_prices)} 条")


# 运行回测
print("  [5.3] 运行回测...")
backtest_engine = BacktestEngine(
    initial_capital=1000000,
    commission_rate=0.0005,
    slippage_rate=0.0001,
    max_position=100000,
    max_capital_usage=0.05
)

equity_curve = backtest_engine.run_backtest(
    spread_prices,
    signals,
    price_col='close',
    volatility_col='volatility'
)
print(f"    ✓ 回测完成，最终权益: ${equity_curve['equity'].iloc[-1]:,.2f}")

# 获取交易日志
trade_log = backtest_engine.get_trade_log()
print(f"    ✓ 总交易次数: {len(trade_log)}")

# 绩效分析
print("  [5.4] 绩效分析...")
analyzer = PerformanceAnalyzer(
    equity_curve,
    initial_capital=1000000,
    risk_free_rate=0.02
)

performance_report = analyzer.generate_performance_report(trade_log)
print(f"    ✓ 总收益率: {performance_report.get('total_return', 0)*100:.2f}%")
print(f"    ✓ 夏普比率: {performance_report.get('sharpe_ratio', 0):.2f}")
print(f"    ✓ 最大回撤: {performance_report.get('max_drawdown', 0)*100:.2f}%")

print("✓ 回测完成")
logger.info("回测完成\n")

# 🔍 调试点6：在此处设置断点，检查回测结果
# 可以查看: equity_curve.tail(), trade_log.head(), performance_report

# ============================================================
# 步骤6：可视化
# ============================================================
print("\n[7/9] 开始可视化...")
logger.info("\n" + "="*60)
logger.info("步骤6：结果可视化")
logger.info("="*60)

print("  [6.1] 生成价格和价差图...")
visualizer.plot_price_and_spread(
    price_data,
    spread_data['CRACK_3_2_1'],
    title='Crack Spread 3:2:1'
)
print("    ✓ price_spread_chart.png")

print("  [6.2] 生成权益曲线图...")
visualizer.plot_equity_curve(equity_curve)
print("    ✓ equity_curve.png")

print("  [6.3] 生成收益率分布图...")
returns = equity_curve['equity'].pct_change().dropna()
visualizer.plot_returns_distribution(returns)
print("    ✓ returns_distribution.png")

print("  [6.4] 生成月度收益热力图...")
visualizer.plot_monthly_returns_heatmap(equity_curve)
print("    ✓ monthly_returns_heatmap.png")

print("  [6.5] 生成滚动指标图...")
visualizer.plot_rolling_metrics(equity_curve, window=60)
print("    ✓ rolling_metrics.png")

print("  [6.6] 生成交易分析图...")
visualizer.plot_trade_analysis(trade_log)
print("    ✓ trade_analysis.png")

print("✓ 可视化完成")
logger.info("可视化完成\n")

# ============================================================
# 完成
# ============================================================
print("\n" + "="*60)
print("策略执行完成！")
print("="*60)
print("\n生成的文件:")
print("  📁 data/trading_data.db          - 数据库")
print("  📁 models/crack_spread_model.pkl - 模型文件")
print("  📁 outputs/charts/*.png          - 图表文件")
print("  📁 logs/debug_strategy.log       - 日志文件")

print("\n可用的全局变量（用于调试）:")
print("  数据相关:")
print("    - price_data       : 期货价格数据字典")
print("    - spread_data      : 价差数据字典")
print("    - macro_data       : 宏观数据字典")
print("    - fundamental_data : 基本面数据字典")
print("\n  特征相关:")
print("    - spread_features  : 价差特征DataFrame")
print("    - price_features   : 价格特征DataFrame")
print("    - technical_features: 技术指标特征DataFrame")
print("    - seasonal_features: 季节性特征DataFrame")
print("    - macro_features   : 宏观特征DataFrame")
print("    - target_df        : 目标变量DataFrame")
print("    - features_df      : 合并后的完整特征DataFrame")
print("\n  模型相关:")
print("    - model            : 训练好的模型")
print("    - X_train, X_test  : 训练/测试特征")
print("    - y_train, y_test  : 训练/测试标签")
print("    - selected_features: 选择的特征列表")
print("\n  回测相关:")
print("    - signals          : 交易信号Series")
print("    - equity_curve     : 权益曲线DataFrame")
print("    - trade_log        : 交易日志DataFrame")
print("    - performance_report: 绩效报告字典")

print("\n💡 调试提示:")
print("  1. 在VS Code中打开此文件")
print("  2. 点击行号左侧设置断点（蓝点）")
print("  3. 按F5或点击'运行和调试'启动调试")
print("  4. 在'变量'面板查看所有变量的值")
print("  5. 在'调试控制台'输入变量名查看详细信息")
print("  例如: price_data.keys(), features_df.shape, model.feature_importance")

logger.info("\n" + "="*60)
logger.info("所有任务完成！")
logger.info("="*60)

# 🔍 最终调试点：程序结束前，所有变量都已计算完成
# 现在可以检查任何变量的最终状态

INFO:src.core.ml_models:使用滚动分位数模式: window=120, 上分位数=0.8, 下分位数=0.2


INFO:src.core.ml_models:滚动分位数统计:
INFO:src.core.ml_models:  上阈值范围: [-1.8829, 1.1266], 均值: -0.4022
INFO:src.core.ml_models:  下阈值范围: [-5.1249, -0.9299], 均值: -3.4477
INFO:src.core.ml_models:回归信号统计:
INFO:src.core.ml_models:  上阈值范围: [-1.8829, 1.1266], 均值: -0.4022
INFO:src.core.ml_models:  下阈值范围: [-5.1249, -0.9299], 均值: -3.4477
INFO:src.core.ml_models:回归信号统计:
INFO:src.core.ml_models:  预测值范围: [-10.0904, 2.8069]
INFO:src.core.ml_models:  做多信号(1): 187 (19.9%)
INFO:src.core.ml_models:  观望信号(0): 551 (58.5%)
INFO:src.core.ml_models:  做空信号(-1): 204 (21.7%)
INFO:src.core.ml_models:生成交易信号完成，信号分布:
INFO:src.core.ml_models: 0    551
-1    204
 1    187
Name: signal, dtype: int64
INFO:src.core.ml_models:  预测值范围: [-10.0904, 2.8069]
INFO:src.core.ml_models:  做多信号(1): 187 (19.9%)
INFO:src.core.ml_models:  观望信号(0): 551 (58.5%)
INFO:src.core.ml_models:  做空信号(-1): 204 (21.7%)
INFO:src.core.ml_models:生成交易信号完成，信号分布:
INFO:src.core.ml_models: 0    551
-1    204
 1    187
Name: signal, dtype: int64
INFO:src.core.ml_

  [5.1] 生成交易信号...


INFO:src.core.ml_models: 0    358
-1    296
 1    288
Name: signal, dtype: int64
INFO:src.core.backtest:开始运行回测...
INFO:src.core.backtest:开始运行回测...
INFO:src.core.backtest:初始资金: $1,000,000.00
INFO:src.core.backtest:手续费率: 0.050%
INFO:src.core.backtest:滑点率: 0.010%
INFO:src.core.backtest:杠杆倍数: 10.0x
INFO:src.core.backtest:保证金比例: 10.0%
INFO:src.core.backtest:初始资金: $1,000,000.00
INFO:src.core.backtest:手续费率: 0.050%
INFO:src.core.backtest:滑点率: 0.010%
INFO:src.core.backtest:杠杆倍数: 10.0x
INFO:src.core.backtest:保证金比例: 10.0%
INFO:src.core.backtest:最大资金使用率: 5%
INFO:src.core.backtest:回测完成，共执行 633 笔交易
INFO:src.core.backtest:最终权益: $1,502,140.62
INFO:src.core.backtest:最大资金使用率: 5%
INFO:src.core.backtest:回测完成，共执行 633 笔交易
INFO:src.core.backtest:最终权益: $1,502,140.62
INFO:src.core.backtest:
INFO:src.core.backtest:绩效分析报告
INFO:src.core.backtest:============================================================
INFO:src.core.backtest:总收益率: 50.21%
INFO:src.core.backtest:年化收益率: 11.76%
INFO:src.core.backtest:年化波动率: 21.46%

    ✓ 生成 942 个信号
    信号分布: {(-10.09039265473816, -0.7382836975256055, -4.830278952598018, -1, 1.0): 1, (-0.9915193314258061, 0.306340097805022, -1.1224538813598712, -1, 0.0): 1, (-1.0512732811875185, 0.28914318412519796, -1.116319425615243, -1, 0.0): 1, (-1.0495112220381517, -1.2297117404111535, -4.2140418890999785, 1, 0.06038223299529158): 1, (-1.04277535335954, -0.17718261498448504, -3.0481430896847725, 0, 0.0): 1, (-1.0390870357799484, -0.9089955211802978, -4.970484497706835, -1, 0.0): 1, (-1.029625561030455, 0.28914318412519796, -1.0339551050618676, -1, 0.0): 1, (-1.0252666883296628, 0.17739796551128076, -1.354347625985864, 1, 0.0): 1, (-1.0250984585371823, -0.4318072699255383, -3.9811418038720383, 1, 0.0): 1, (-1.0172445123483946, -0.17718261498448504, -2.7178958271608353, 0, 0.0): 1, (-1.0144908899585519, 0.21625826943007914, -0.9718492437043571, -1, 0.035890393573640995): 1, (-1.0054340431597866, -1.54531353944669, -4.38474739464301, 1, 0.19013631724468394): 1, (-0.9969556072771

...

    ✓ monthly_returns_heatmap.png
  [6.5] 生成滚动指标图...


INFO:src.core.visualization:滚动指标图表已保存: outputs/charts/rolling_metrics.png


    ✓ rolling_metrics.png
  [6.6] 生成交易分析图...


INFO:src.core.visualization:交易分析图表已保存: outputs/charts/trade_analysis.png
INFO:__main__:可视化完成

INFO:__main__:
INFO:__main__:所有任务完成！
INFO:__main__:============================================================
INFO:__main__:可视化完成

INFO:__main__:
INFO:__main__:所有任务完成！
INFO:__main__:============================================================


    ✓ trade_analysis.png
✓ 可视化完成

策略执行完成！

生成的文件:
  📁 data/trading_data.db          - 数据库
  📁 models/crack_spread_model.pkl - 模型文件
  📁 outputs/charts/*.png          - 图表文件
  📁 logs/debug_strategy.log       - 日志文件

可用的全局变量（用于调试）:
  数据相关:
    - price_data       : 期货价格数据字典
    - spread_data      : 价差数据字典
    - macro_data       : 宏观数据字典
    - fundamental_data : 基本面数据字典

  特征相关:
    - spread_features  : 价差特征DataFrame
    - price_features   : 价格特征DataFrame
    - technical_features: 技术指标特征DataFrame
    - seasonal_features: 季节性特征DataFrame
    - macro_features   : 宏观特征DataFrame
    - target_df        : 目标变量DataFrame
    - features_df      : 合并后的完整特征DataFrame

  模型相关:
    - model            : 训练好的模型
    - X_train, X_test  : 训练/测试特征
    - y_train, y_test  : 训练/测试标签
    - selected_features: 选择的特征列表

  回测相关:
    - signals          : 交易信号Series
    - equity_curve     : 权益曲线DataFrame
    - trade_log        : 交易日志DataFrame
    - performance_report: 绩效报告字典

💡 调试提示:
  1. 在VS Code中打开此文件
  2. 点击行号左侧设置断点（蓝

# 超参数优化

使用不同的搜索方法优化模型超参数：
- 网格搜索（Grid Search）：遍历所有参数组合
- 随机搜索（Random Search）：随机采样参数组合
- 贝叶斯优化（Bayesian Optimization）：智能搜索最优参数

## 步骤：
1. 导入超参数优化模块
2. 选择搜索方法
3. 执行参数搜索
4. 比较不同方法的结果
5. 使用最佳参数重新训练模型

In [ ]:
# 导入超参数优化模块
from src.core.hyperparameter_tuning import HyperparameterTuner

# 创建超参数优化器
tuner = HyperparameterTuner(
    model_type='gradient_boosting',
    task='classification',
    scoring='f1_weighted',  # 使用加权F1分数
    cv=5,  # 5折交叉验证
    n_jobs=-1,  # 使用所有CPU核心
    verbose=1
)

print("超参数优化器初始化完成")
print(f"模型类型: {tuner.model_type}")
print(f"评分指标: {tuner.scoring}")
print(f"交叉验证折数: {tuner.cv}")

## 方法1：网格搜索（Grid Search）

网格搜索会遍历所有可能的参数组合，找到最优参数。

**优点**：能找到全局最优解（在给定的参数空间内）  
**缺点**：计算成本高，参数组合数呈指数增长  
**适用场景**：参数空间较小，计算资源充足

In [21]:
# 方法1：网格搜索
print("\n" + "="*60)
print("方法1：网格搜索（Grid Search）")
print("="*60)

# 创建基础模型
from sklearn.ensemble import GradientBoostingClassifier
from src.core.hyperparameter_tuning import HyperparameterTuner
base_model_grid = GradientBoostingClassifier(random_state=42)

tuner=HyperparameterTuner(
    model_type='gradient_boosting',
    task='classification',
    cv=5,)
# 执行网格搜索
best_params_grid = tuner.grid_search(
    model=base_model_grid,
    X_train=X_train,
    y_train=y_train
)

print("\n网格搜索结果:")
print(f"最佳参数: {best_params_grid}")
print(f"最佳得分: {tuner.best_score_:.4f}")

# 显示前10个最佳参数组合
print("\n前10个最佳参数组合:")
summary_grid = tuner.get_search_results_summary()
print(summary_grid.head(10).to_string())

INFO:src.core.hyperparameter_tuning:开始网格搜索优化 gradient_boosting...



方法1：网格搜索（Grid Search）
Fitting 5 folds for each of 243 candidates, totalling 1215 fits


INFO:src.core.hyperparameter_tuning:网格搜索完成，耗时: 3564.19秒
INFO:src.core.hyperparameter_tuning:最佳参数: {'learning_rate': 0.01, 'max_depth': 3, 'min_samples_split': 5, 'n_estimators': 50, 'subsample': 0.8}
INFO:src.core.hyperparameter_tuning:最佳得分: 0.4088



网格搜索结果:
最佳参数: {'learning_rate': 0.01, 'max_depth': 3, 'min_samples_split': 5, 'n_estimators': 50, 'subsample': 0.8}
最佳得分: 0.4088

前10个最佳参数组合:
   param_learning_rate param_max_depth param_min_samples_split param_n_estimators param_subsample  mean_test_score  std_test_score  mean_train_score  std_train_score  mean_fit_time  rank_test_score
9                 0.01               3                       5                 50             0.8         0.408765        0.038537          0.550647         0.013998      11.337362                1
18                0.01               3                      10                 50             0.8         0.408566        0.038562          0.550647         0.014007      11.204950                2
0                 0.01               3                       2                 50             0.8         0.407968        0.038661          0.550598         0.013967      12.368808                3
1                 0.01               3                       2   

## 方法2：随机搜索（Random Search）

随机搜索从参数空间中随机采样固定数量的参数组合。

**优点**：计算成本可控，通常能找到接近最优的解  
**缺点**：不保证找到全局最优解  
**适用场景**：参数空间大，需要快速获得较好结果

In [ ]:
# # 方法2：随机搜索
# print("\n" + "="*60)
# print("方法2：随机搜索（Random Search）")
# print("="*60)

# # 创建新的优化器和基础模型
# tuner_random = HyperparameterTuner(
#     model_type='gradient_boosting',
#     task='classification',
#     scoring='f1_weighted',
#     cv=5,
#     n_jobs=-1,
#     verbose=1
# )

# base_model_random = GradientBoostingClassifier(random_state=42)

# # 执行随机搜索（50次迭代）
# best_params_random = tuner_random.random_search(
#     model=base_model_random,
#     X_train=X_train,
#     y_train=y_train,
#     n_iter=50,
#     random_state=42
# )

# print("\n随机搜索结果:")
# print(f"最佳参数: {best_params_random}")
# print(f"最佳得分: {tuner_random.best_score_:.4f}")

# # 显示前10个最佳参数组合
# print("\n前10个最佳参数组合:")
# summary_random = tuner_random.get_search_results_summary()
# print(summary_random.head(10).to_string())

## 方法3：贝叶斯优化（Bayesian Optimization）

贝叶斯优化使用概率模型智能地选择下一个要尝试的参数组合。

**优点**：通常能以较少的迭代次数找到最优解  
**缺点**：需要额外安装 scikit-optimize 库  
**适用场景**：参数空间大，希望高效找到最优解

> 注意：需要先安装：`pip install scikit-optimize`

In [ ]:
# # 方法3：贝叶斯优化（可选）
# print("\n" + "="*60)
# print("方法3：贝叶斯优化（Bayesian Optimization）")
# print("="*60)

# try:
#     # 创建新的优化器和基础模型
#     tuner_bayes = HyperparameterTuner(
#         model_type='gradient_boosting',
#         task='classification',
#         scoring='f1_weighted',
#         cv=5,
#         n_jobs=-1,
#         verbose=1
#     )
    
#     base_model_bayes = GradientBoostingClassifier(random_state=42)
    
#     # 执行贝叶斯优化（30次迭代）
#     best_params_bayes = tuner_bayes.bayesian_search(
#         model=base_model_bayes,
#         X_train=X_train,
#         y_train=y_train,
#         n_iter=30,
#         random_state=42
#     )
    
#     print("\n贝叶斯优化结果:")
#     print(f"最佳参数: {best_params_bayes}")
#     print(f"最佳得分: {tuner_bayes.best_score_:.4f}")
    
#     # 显示前10个最佳参数组合
#     print("\n前10个最佳参数组合:")
#     summary_bayes = tuner_bayes.get_search_results_summary()
#     print(summary_bayes.head(10).to_string())
    
# except ImportError as e:
#     print("⚠️  贝叶斯优化不可用")
#     print("请安装 scikit-optimize: pip install scikit-optimize")
#     print(f"错误信息: {e}")
#     tuner_bayes = None
#     best_params_bayes = None

## 比较搜索方法

比较不同搜索方法的性能，包括最佳得分和运行时间。

In [ ]:
# # 比较所有搜索方法
# print("\n" + "="*60)
# print("搜索方法比较")
# print("="*60)

# # 收集结果
# methods_results = {
#     'Grid Search': {
#         'best_score': tuner.best_score_,
#         'time': tuner.search_history_[-1]['time'],
#         'n_evaluations': tuner.search_history_[-1].get('n_combinations', 'N/A')
#     },
#     'Random Search': {
#         'best_score': tuner_random.best_score_,
#         'time': tuner_random.search_history_[-1]['time'],
#         'n_evaluations': tuner_random.search_history_[-1].get('n_iterations', 'N/A')
#     }
# }

# if tuner_bayes is not None:
#     methods_results['Bayesian Optimization'] = {
#         'best_score': tuner_bayes.best_score_,
#         'time': tuner_bayes.search_history_[-1]['time'],
#         'n_evaluations': tuner_bayes.search_history_[-1].get('n_iterations', 'N/A')
#     }

# # 创建比较表
# comparison_df = pd.DataFrame(methods_results).T
# comparison_df = comparison_df.sort_values('best_score', ascending=False)

# print("\n搜索方法性能比较:")
# print(comparison_df.to_string())

# # 找出最佳方法
# best_method = comparison_df['best_score'].idxmax()
# print(f"\n🏆 最佳方法: {best_method}")
# print(f"   最佳得分: {comparison_df.loc[best_method, 'best_score']:.4f}")
# print(f"   运行时间: {comparison_df.loc[best_method, 'time']:.2f}秒")

# # 可视化比较
# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# # 得分比较
# ax1 = axes[0]
# comparison_df['best_score'].plot(kind='bar', ax=ax1, color=['#2ecc71', '#3498db', '#e74c3c'])
# ax1.set_title('Best Score Comparison', fontsize=14, fontweight='bold')
# ax1.set_ylabel('Score', fontsize=12)
# ax1.set_xlabel('Method', fontsize=12)
# ax1.grid(True, alpha=0.3, axis='y')
# ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right')

# # 时间比较
# ax2 = axes[1]
# comparison_df['time'].plot(kind='bar', ax=ax2, color=['#2ecc71', '#3498db', '#e74c3c'])
# ax2.set_title('Execution Time Comparison', fontsize=14, fontweight='bold')
# ax2.set_ylabel('Time (seconds)', fontsize=12)
# ax2.set_xlabel('Method', fontsize=12)
# ax2.grid(True, alpha=0.3, axis='y')
# ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')

# plt.tight_layout()
# plt.savefig('outputs/charts/hyperparameter_tuning_comparison.png', dpi=300, bbox_inches='tight')
# print("\n✓ 比较图已保存: outputs/charts/hyperparameter_tuning_comparison.png")
# plt.show()

## 使用最佳参数重新训练模型

使用搜索到的最佳参数重新训练模型，并评估性能提升。

In [22]:
# 使用最佳参数重新训练模型
print("\n" + "="*60)
print("使用最佳参数重新训练模型")
print("="*60)
best_params_final = best_params_grid
best_tuner = tuner
# 选择最佳方法的参数
# if best_method == 'Grid Search':
#     best_params_final = best_params_grid
#     best_tuner = tuner
# elif best_method == 'Random Search':
#     best_params_final = best_params_random
#     best_tuner = tuner_random
# else:
#     best_params_final = best_params_bayes
#     best_tuner = tuner_bayes

# print(f"\n使用 {best_method} 的最佳参数:")
for param, value in best_params_final.items():
    print(f"  {param}: {value}")

# 使用最佳参数创建新模型
model_optimized = MLModel(model_type='gradient_boosting', task='classification')

# 使用最佳参数训练
print("\n训练优化后的模型...")
model_optimized.model = GradientBoostingClassifier(**best_params_final, random_state=42)
model_optimized.model.fit(X_train, y_train)

# 评估优化后的模型
metrics_optimized = model_optimized.evaluate(X_test, y_test)

print("\n优化后模型性能:")
print(f"  准确率: {metrics_optimized.get('accuracy', 0):.4f}")
print(f"  F1分数: {metrics_optimized.get('f1', 0):.4f}")
print(f"  精确率: {metrics_optimized.get('precision', 0):.4f}")
print(f"  召回率: {metrics_optimized.get('recall', 0):.4f}")

# 与原始模型比较
print("\n性能对比:")
print(f"  原始模型 F1: {metrics.get('f1', 0):.4f}")
print(f"  优化模型 F1: {metrics_optimized.get('f1', 0):.4f}")
improvement = (metrics_optimized.get('f1', 0) - metrics.get('f1', 0)) / metrics.get('f1', 1) * 100
print(f"  提升幅度: {improvement:+.2f}%")

# 保存优化结果
best_tuner.save_results('models/tuning_results')
print("\n✓ 优化结果已保存到 models/tuning_results/")

# 更新全局model变量为优化后的模型
model = model_optimized
print("\n✓ 全局模型已更新为优化后的模型")


使用最佳参数重新训练模型
  learning_rate: 0.01
  max_depth: 3
  min_samples_split: 5
  n_estimators: 50
  subsample: 0.8

训练优化后的模型...


INFO:src.core.ml_models:
模型评估结果:
INFO:src.core.ml_models:准确率: 0.3710
INFO:src.core.ml_models:精确率: 0.3432
INFO:src.core.ml_models:召回率: 0.3710
INFO:src.core.ml_models:F1分数: 0.3234
INFO:src.core.ml_models:
分类报告:
INFO:src.core.ml_models:              precision    recall  f1-score   support

          -1       0.37      0.19      0.25       337
           0       0.39      0.69      0.50       519
           1       0.26      0.11      0.16       400

    accuracy                           0.37      1256
   macro avg       0.34      0.33      0.30      1256
weighted avg       0.34      0.37      0.32      1256

INFO:src.core.ml_models:
混淆矩阵:
INFO:src.core.ml_models:[[ 63 238  36]
 [ 68 358  93]
 [ 41 314  45]]
INFO:src.core.hyperparameter_tuning:最佳参数已保存: models\tuning_results\gradient_boosting_best_params.json
INFO:src.core.hyperparameter_tuning:搜索历史已保存: models\tuning_results\gradient_boosting_search_history.json
INFO:src.core.hyperparameter_tuning:搜索结果已保存: models\tuning_results\gradient_bo


优化后模型性能:
  准确率: 0.3710
  F1分数: 0.3234
  精确率: 0.3432
  召回率: 0.3710

性能对比:
  原始模型 F1: 0.3280
  优化模型 F1: 0.3234
  提升幅度: -1.41%

✓ 优化结果已保存到 models/tuning_results/

✓ 全局模型已更新为优化后的模型


In [16]:
signal_generator = SignalGenerator(model, threshold=0.5, signal_holding_days=20)
signals = signal_generator.generate_signals(X_test, use_probability=True)
signals.index = test_idx
print(f"    ✓ 生成 {len(signals)} 个信号")
print(f"    信号分布: {signals.value_counts().to_dict()}")

# 获取价差价格数据
print("  [5.2] 准备价格数据...")
# 确保spread_data索引与test_idx时区一致
spread_df_for_backtest = spread_data['CRACK_3_2_1'].copy()
if hasattr(spread_df_for_backtest.index, 'tz') and spread_df_for_backtest.index.tz is not None:
    spread_df_for_backtest.index = spread_df_for_backtest.index.tz_localize(None)

spread_prices = spread_df_for_backtest.loc[test_idx, ['spread']].copy()
spread_prices.columns = ['close']
spread_prices['volatility'] = spread_prices['close'].pct_change().rolling(20).std()
print(f"    ✓ 价格数据: {len(spread_prices)} 条")

# 运行回测
print("  [5.3] 运行回测...")
backtest_engine = BacktestEngine(
    initial_capital=1000000,
    commission_rate=0.0005,
    slippage_rate=0.0001,
    max_position=1e16,
    max_capital_usage=0.05
)

equity_curve = backtest_engine.run_backtest(
    spread_prices,
    signals,
    price_col='close',
    volatility_col='volatility'
)
print(f"    ✓ 回测完成，最终权益: ${equity_curve['equity'].iloc[-1]:,.2f}")

# 获取交易日志
trade_log = backtest_engine.get_trade_log()
print(f"    ✓ 总交易次数: {len(trade_log)}")

# 绩效分析
print("  [5.4] 绩效分析...")
analyzer = PerformanceAnalyzer(
    equity_curve,
    initial_capital=1000000,
    risk_free_rate=0.02
)

performance_report = analyzer.generate_performance_report(trade_log)
print(f"    ✓ 总收益率: {performance_report.get('total_return', 0)*100:.2f}%")
print(f"    ✓ 夏普比率: {performance_report.get('sharpe_ratio', 0):.2f}")
print(f"    ✓ 最大回撤: {performance_report.get('max_drawdown', 0)*100:.2f}%")

print("✓ 回测完成")
logger.info("回测完成\n")

# 🔍 调试点6：在此处设置断点，检查回测结果
# 可以查看: equity_curve.tail(), trade_log.head(), performance_report

# ============================================================
# 步骤6：可视化
# ============================================================
print("\n[7/9] 开始可视化...")
logger.info("\n" + "="*60)
logger.info("步骤6：结果可视化")
logger.info("="*60)

print("  [6.1] 生成价格和价差图...")
visualizer.plot_price_and_spread(
    price_data,
    spread_data['CRACK_3_2_1'],
    title='Crack Spread 3:2:1'
)
print("    ✓ price_spread_chart.png")

print("  [6.2] 生成权益曲线图...")
visualizer.plot_equity_curve(equity_curve)
print("    ✓ equity_curve.png")

print("  [6.3] 生成收益率分布图...")
returns = equity_curve['equity'].pct_change().dropna()
visualizer.plot_returns_distribution(returns)
print("    ✓ returns_distribution.png")

print("  [6.4] 生成月度收益热力图...")
visualizer.plot_monthly_returns_heatmap(equity_curve)
print("    ✓ monthly_returns_heatmap.png")

print("  [6.5] 生成滚动指标图...")
visualizer.plot_rolling_metrics(equity_curve, window=60)
print("    ✓ rolling_metrics.png")

print("  [6.6] 生成交易分析图...")
visualizer.plot_trade_analysis(trade_log)
print("    ✓ trade_analysis.png")

print("✓ 可视化完成")
logger.info("可视化完成\n")

INFO:src.core.ml_models:生成交易信号完成，信号分布:
INFO:src.core.ml_models:-1    851
 0    289
 1    116
Name: signal, dtype: int64
INFO:src.core.ml_models:-1    851
 0    289
 1    116
Name: signal, dtype: int64
INFO:src.core.ml_models:应用20天信号维持后，信号分布:
INFO:src.core.ml_models:-1    1094
 1     160
 0       2
Name: signal, dtype: int64
INFO:src.core.backtest:开始运行回测...
INFO:src.core.ml_models:应用20天信号维持后，信号分布:
INFO:src.core.ml_models:-1    1094
 1     160
 0       2
Name: signal, dtype: int64
INFO:src.core.backtest:开始运行回测...
INFO:src.core.backtest:初始资金: $1,000,000.00
INFO:src.core.backtest:手续费率: 0.050%
INFO:src.core.backtest:滑点率: 0.010%
INFO:src.core.backtest:杠杆倍数: 10.0x
INFO:src.core.backtest:保证金比例: 10.0%
INFO:src.core.backtest:最大资金使用率: 5%
INFO:src.core.backtest:初始资金: $1,000,000.00
INFO:src.core.backtest:手续费率: 0.050%
INFO:src.core.backtest:滑点率: 0.010%
INFO:src.core.backtest:杠杆倍数: 10.0x
INFO:src.core.backtest:保证金比例: 10.0%
INFO:src.core.backtest:最大资金使用率: 5%
INFO:src.core.backtest:回测完成，共执行 1234 笔交易
IN

    ✓ 生成 1256 个信号
    信号分布: {(-1, 0.38574165088606793, 0.23979555234098465, 0.37446279677294736, 0.38574165088606793, -1, 0.38574165088606793): 1, (-1, 0.9991399023125428, 0.0008245369770305406, 3.556071042667221e-05, 0.9991399023125428, -1, 0.9991399023125428): 1, (-1, 0.9992705849993667, 0.0005492758369901867, 0.00018013916364295463, 0.9992705849993667, -1, 0.9992705849993667): 1, (-1, 0.9992324435421083, 0.0007325913292888901, 3.4965128602915604e-05, 0.9992324435421083, -1, 0.9992324435421083): 1, (-1, 0.9991910854187077, 0.0007150793386436705, 9.383524264840007e-05, 0.9991910854187077, -1, 0.9991910854187077): 1, (-1, 0.9991687914006406, 0.0007961724159898763, 3.503618336954858e-05, 0.9991687914006406, -1, 0.9991687914006406): 1, (-1, 0.999161187230139, 0.0008339904773467344, 4.822292514693809e-06, 0.999161187230139, -1, 0.999161187230139): 1, (-1, 0.9991609904167106, 0.000291916053960869, 0.0005470935293287259, 0.9991609904167106, -1, 0.9991609904167106): 1, (-1, 0.999151403452987

INFO:src.core.visualization:价格和价差图表已保存: outputs/charts/price_spread_chart.png


    ✓ price_spread_chart.png
  [6.2] 生成权益曲线图...


INFO:src.core.visualization:权益曲线图表已保存: outputs/charts/equity_curve.png


...

    ✓ monthly_returns_heatmap.png
  [6.5] 生成滚动指标图...


INFO:src.core.visualization:滚动指标图表已保存: outputs/charts/rolling_metrics.png


    ✓ rolling_metrics.png
  [6.6] 生成交易分析图...


INFO:src.core.visualization:交易分析图表已保存: outputs/charts/trade_analysis.png
INFO:__main__:可视化完成

INFO:__main__:可视化完成



    ✓ trade_analysis.png
✓ 可视化完成


In [11]:
# ============================================================
# 诊断杠杆使用情况
# ============================================================
import pandas as pd
import numpy as np

# 检查权益曲线
equity_df = pd.DataFrame(backtest_engine.equity_curve)

print("=" * 80)
print("杠杆使用情况诊断")
print("=" * 80)

print(f"\n📊 基本配置:")
print(f"  初始资金: ${backtest_engine.initial_capital:,.2f}")
print(f"  杠杆倍数: {backtest_engine.leverage}x")
print(f"  保证金比例: {backtest_engine.margin_ratio * 100:.1f}%")
print(f"  最大资金使用率: {backtest_engine.max_capital_usage * 100:.0f}%")
print(f"  最大持仓限制: {backtest_engine.max_position} 手")

print(f"\n📈 回测结果:")
print(f"  最终权益: ${equity_df['equity'].iloc[-1]:,.2f}")
print(f"  总收益率: {(equity_df['equity'].iloc[-1] / backtest_engine.initial_capital - 1) * 100:.2f}%")
print(f"  总交易次数: {len(backtest_engine.trades)}")

print(f"\n💰 保证金使用统计:")
print(f"  峰值保证金占用: ${equity_df['margin_used'].max():,.2f}")
print(f"  平均保证金占用: ${equity_df['margin_used'].mean():,.2f}")
print(f"  峰值占用率: {equity_df['margin_usage_rate'].max() * 100:.2f}%")
print(f"  平均占用率: {equity_df['margin_usage_rate'].mean() * 100:.2f}%")

print(f"\n📍 持仓统计:")
print(f"  最大持仓: {equity_df['position'].abs().max()} 手")
print(f"  平均持仓: {equity_df['position'].abs().mean():.2f} 手")
print(f"  持仓时间占比: {(equity_df['position'] != 0).sum() / len(equity_df) * 100:.1f}%")

print(f"\n🎯 实际杠杆率统计:")
print(f"  峰值杠杆: {equity_df['leverage_ratio'].max():.2f}x")
print(f"  平均杠杆: {equity_df[equity_df['position'] != 0]['leverage_ratio'].mean():.2f}x")

print(f"\n📊 问题诊断:")

# 检查是否受持仓限制
max_position_reached = (equity_df['position'].abs() >= backtest_engine.max_position).sum()
if max_position_reached > 0:
    print(f"  ⚠️ 有 {max_position_reached} 天达到了最大持仓限制 ({backtest_engine.max_position}手)")
    print(f"  ⚠️ 持仓限制阻止了杠杆充分利用！")

# 检查价格水平
avg_price = spread_prices['close'].mean()
print(f"\n  平均价差价格: ${avg_price:.2f}")

# 计算理论最大持仓
available_for_trading = backtest_engine.initial_capital * backtest_engine.max_capital_usage
max_contract_value = available_for_trading * backtest_engine.leverage
theoretical_max_position = int(max_contract_value / avg_price)
print(f"\n  理论最大持仓计算:")
print(f"    可用资金: ${available_for_trading:,.2f}")
print(f"    杠杆放大: {backtest_engine.leverage}x")
print(f"    可控制价值: ${max_contract_value:,.2f}")
print(f"    理论最大持仓: {theoretical_max_position:,} 手")
print(f"    实际最大持仓: {backtest_engine.max_position} 手")
print(f"    持仓使用率: {backtest_engine.max_position / theoretical_max_position * 100:.2f}%")

# 计算如果去掉持仓限制，收益会如何变化
if theoretical_max_position > backtest_engine.max_position:
    amplification = theoretical_max_position / backtest_engine.max_position
    estimated_return = (equity_df['equity'].iloc[-1] / backtest_engine.initial_capital - 1) * amplification
    print(f"\n  💡 如果去掉持仓限制:")
    print(f"    持仓放大倍数: {amplification:.2f}x")
    print(f"    预估收益率: {estimated_return * 100:.2f}%")
    
    # 波动率也会成倍放大
    current_volatility = equity_df['equity'].pct_change().std()
    estimated_volatility = current_volatility * amplification
    print(f"    当前日波动率: {current_volatility * 100:.4f}%")
    print(f"    预估日波动率: {estimated_volatility * 100:.4f}%")

print(f"\n🔧 解决方案:")
print(f"  方案1: 提高 max_position 到 {theoretical_max_position:,} 手")
print(f"  方案2: 降低杠杆到 {backtest_engine.leverage * backtest_engine.max_position / theoretical_max_position:.1f}x")
print(f"  方案3: 使用动态持仓（基于资金百分比而非固定数量）")

print("\n" + "=" * 80)

杠杆使用情况诊断

📊 基本配置:
  初始资金: $1,000,000.00
  杠杆倍数: 10.0x
  保证金比例: 10.0%
  最大资金使用率: 80%
  最大持仓限制: 100 手

📈 回测结果:
  最终权益: $1,004,337.24
  总收益率: 0.43%
  总交易次数: 608

💰 保证金使用统计:
  峰值保证金占用: $811.39
  平均保证金占用: $521.92
  峰值占用率: 0.08%
  平均占用率: 0.05%

📍 持仓统计:
  最大持仓: 86 手
  平均持仓: 70.50 手
  持仓时间占比: 99.9%

🎯 实际杠杆率统计:
  峰值杠杆: 0.01x
  平均杠杆: 0.01x

📊 问题诊断:

  平均价差价格: $80.99

  理论最大持仓计算:
    可用资金: $800,000.00
    杠杆放大: 10.0x
    可控制价值: $8,000,000.00
    理论最大持仓: 98,777 手
    实际最大持仓: 100 手
    持仓使用率: 0.10%

  💡 如果去掉持仓限制:
    持仓放大倍数: 987.77x
    预估收益率: 428.42%
    当前日波动率: 0.0298%
    预估日波动率: 29.4818%

🔧 解决方案:
  方案1: 提高 max_position 到 98,777 手
  方案2: 降低杠杆到 0.0x
  方案3: 使用动态持仓（基于资金百分比而非固定数量）



## 🔧 解决方案：使用动态持仓充分利用杠杆

**问题分析**：
- 当前使用固定的 `max_position=100` 手，导致杠杆无法发挥作用
- 理论上可以开 98,777 手，但被限制在 100 手
- 持仓仅使用了 0.10% 的可用资金

**解决方案**：
1. **方案A**：提高 max_position 到理论值（98,777手）- ⚠️ 风险极高
2. **方案B**：使用动态持仓（推荐）- 基于资金百分比自动调整
3. **方案C**：调整杠杆和持仓的平衡配置

下面我们测试**方案B：动态持仓**

In [12]:
# ============================================================
# 方案对比：不同持仓配置的回测
# ============================================================

print("\n" + "="*80)
print("杠杆配置对比测试")
print("="*80)

# 配置不同的回测方案
test_configs = [
    {
        'name': '当前配置（持仓限制100手）',
        'max_position': 100,
        'leverage': 10.0,
        'margin_ratio': 0.1,
        'max_capital_usage': 0.8
    },
    {
        'name': '中等持仓（1000手）',
        'max_position': 1000,
        'leverage': 10.0,
        'margin_ratio': 0.1,
        'max_capital_usage': 0.8
    },
    {
        'name': '高持仓（10000手）',
        'max_position': 10000,
        'leverage': 10.0,
        'margin_ratio': 0.1,
        'max_capital_usage': 0.8
    },
    {
        'name': '完全杠杆（50000手）',
        'max_position': 50000,
        'leverage': 10.0,
        'margin_ratio': 0.1,
        'max_capital_usage': 0.8
    }
]

results_comparison = []

for config in test_configs:
    print(f"\n{'='*60}")
    print(f"测试配置: {config['name']}")
    print(f"{'='*60}")
    
    # 创建回测引擎
    engine = BacktestEngine(
        initial_capital=1000000,
        commission_rate=0.0005,
        slippage_rate=0.0001,
        max_position=config['max_position'],
        leverage=config['leverage'],
        margin_ratio=config['margin_ratio'],
        max_capital_usage=config['max_capital_usage']
    )
    
    # 运行回测
    equity_curve_test = engine.run_backtest(
        spread_prices,
        signals,
        price_col='close',
        volatility_col='volatility'
    )
    
    # 计算指标
    equity_df_test = pd.DataFrame(engine.equity_curve)
    final_equity = equity_df_test['equity'].iloc[-1]
    total_return = (final_equity / 1000000 - 1) * 100
    volatility = equity_df_test['equity'].pct_change().std() * np.sqrt(252) * 100  # 年化波动率
    max_drawdown = (equity_df_test['equity'] / equity_df_test['equity'].cummax() - 1).min() * 100
    sharpe = (total_return / 100) / (volatility / 100) if volatility > 0 else 0
    
    # 保证金统计
    peak_margin_usage = equity_df_test['margin_usage_rate'].max() * 100
    avg_position = equity_df_test['position'].abs().mean()
    avg_leverage = equity_df_test[equity_df_test['position'] != 0]['leverage_ratio'].mean()
    
    result = {
        '配置': config['name'],
        '最大持仓': config['max_position'],
        '最终权益': final_equity,
        '总收益率(%)': total_return,
        '年化波动率(%)': volatility,
        '最大回撤(%)': max_drawdown,
        '夏普比率': sharpe,
        '峰值保证金占用(%)': peak_margin_usage,
        '平均持仓': avg_position,
        '平均杠杆': avg_leverage
    }
    
    results_comparison.append(result)
    
    print(f"  最终权益: ${final_equity:,.2f}")
    print(f"  总收益率: {total_return:.2f}%")
    print(f"  年化波动率: {volatility:.2f}%")
    print(f"  最大回撤: {max_drawdown:.2f}%")
    print(f"  夏普比率: {sharpe:.2f}")
    print(f"  平均持仓: {avg_position:.0f} 手")
    print(f"  实际杠杆: {avg_leverage:.2f}x")

# 创建对比表
print("\n" + "="*80)
print("📊 配置对比总结")
print("="*80)

comparison_df = pd.DataFrame(results_comparison)
print(comparison_df.to_string(index=False))

# 可视化对比
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. 收益率对比
ax1 = axes[0, 0]
comparison_df.plot(x='最大持仓', y='总收益率(%)', kind='bar', ax=ax1, 
                   color='#2ecc71', legend=False)
ax1.set_title('总收益率对比', fontsize=14, fontweight='bold')
ax1.set_ylabel('总收益率 (%)', fontsize=12)
ax1.set_xlabel('最大持仓配置', fontsize=12)
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_xticklabels([f"{x:,}" for x in comparison_df['最大持仓']], rotation=45, ha='right')

# 2. 波动率对比
ax2 = axes[0, 1]
comparison_df.plot(x='最大持仓', y='年化波动率(%)', kind='bar', ax=ax2, 
                   color='#e74c3c', legend=False)
ax2.set_title('年化波动率对比', fontsize=14, fontweight='bold')
ax2.set_ylabel('年化波动率 (%)', fontsize=12)
ax2.set_xlabel('最大持仓配置', fontsize=12)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_xticklabels([f"{x:,}" for x in comparison_df['最大持仓']], rotation=45, ha='right')

# 3. 夏普比率对比
ax3 = axes[1, 0]
comparison_df.plot(x='最大持仓', y='夏普比率', kind='bar', ax=ax3, 
                   color='#3498db', legend=False)
ax3.set_title('夏普比率对比', fontsize=14, fontweight='bold')
ax3.set_ylabel('夏普比率', fontsize=12)
ax3.set_xlabel('最大持仓配置', fontsize=12)
ax3.grid(True, alpha=0.3, axis='y')
ax3.axhline(y=0, color='red', linestyle='--', linewidth=1)
ax3.set_xticklabels([f"{x:,}" for x in comparison_df['最大持仓']], rotation=45, ha='right')

# 4. 最大回撤对比
ax4 = axes[1, 1]
comparison_df.plot(x='最大持仓', y='最大回撤(%)', kind='bar', ax=ax4, 
                   color='#f39c12', legend=False)
ax4.set_title('最大回撤对比', fontsize=14, fontweight='bold')
ax4.set_ylabel('最大回撤 (%)', fontsize=12)
ax4.set_xlabel('最大持仓配置', fontsize=12)
ax4.grid(True, alpha=0.3, axis='y')
ax4.set_xticklabels([f"{x:,}" for x in comparison_df['最大持仓']], rotation=45, ha='right')

plt.tight_layout()
plt.savefig('outputs/charts/leverage_comparison.png', dpi=300, bbox_inches='tight')
print("\n✓ 对比图已保存: outputs/charts/leverage_comparison.png")
plt.show()

print("\n" + "="*80)
print("💡 结论和建议")
print("="*80)
print("\n观察结果:")
print("  - 持仓限制越高，收益和波动率都越大")
print("  - 杠杆充分利用后，波动率会显著提升")
print("  - 需要在收益和风险之间找到平衡点")
print("\n建议:")
print("  1. 如果追求稳健：保持 100-1000 手")
print("  2. 如果追求收益：使用 10000-50000 手（但风险极高）")
print("  3. 最佳实践：使用动态持仓管理，根据市场波动率调整")
print("="*80)

INFO:src.core.backtest:开始运行回测...
INFO:src.core.backtest:初始资金: $1,000,000.00
INFO:src.core.backtest:手续费率: 0.050%
INFO:src.core.backtest:滑点率: 0.010%
INFO:src.core.backtest:杠杆倍数: 10.0x
INFO:src.core.backtest:保证金比例: 10.0%
INFO:src.core.backtest:最大资金使用率: 80%
INFO:src.core.backtest:回测完成，共执行 608 笔交易
INFO:src.core.backtest:最终权益: $1,004,337.24
INFO:src.core.backtest:开始运行回测...
INFO:src.core.backtest:初始资金: $1,000,000.00
INFO:src.core.backtest:手续费率: 0.050%
INFO:src.core.backtest:滑点率: 0.010%
INFO:src.core.backtest:杠杆倍数: 10.0x
INFO:src.core.backtest:保证金比例: 10.0%
INFO:src.core.backtest:最大资金使用率: 80%



杠杆配置对比测试

测试配置: 当前配置（持仓限制100手）
  最终权益: $1,004,337.24
  总收益率: 0.43%
  年化波动率: 0.47%
  最大回撤: -0.64%
  夏普比率: 0.92
  平均持仓: 71 手
  实际杠杆: 0.01x

测试配置: 中等持仓（1000手）


INFO:src.core.backtest:回测完成，共执行 1078 笔交易
INFO:src.core.backtest:最终权益: $1,043,501.65
INFO:src.core.backtest:开始运行回测...
INFO:src.core.backtest:初始资金: $1,000,000.00
INFO:src.core.backtest:手续费率: 0.050%
INFO:src.core.backtest:滑点率: 0.010%
INFO:src.core.backtest:杠杆倍数: 10.0x
INFO:src.core.backtest:保证金比例: 10.0%
INFO:src.core.backtest:最大资金使用率: 80%
INFO:src.core.backtest:回测完成，共执行 1217 笔交易
INFO:src.core.backtest:最终权益: $1,435,553.72
INFO:src.core.backtest:开始运行回测...
INFO:src.core.backtest:初始资金: $1,000,000.00
INFO:src.core.backtest:手续费率: 0.050%
INFO:src.core.backtest:滑点率: 0.010%
INFO:src.core.backtest:杠杆倍数: 10.0x
INFO:src.core.backtest:保证金比例: 10.0%
INFO:src.core.backtest:最大资金使用率: 80%


  最终权益: $1,043,501.65
  总收益率: 4.35%
  年化波动率: 4.78%
  最大回撤: -6.35%
  夏普比率: 0.91
  平均持仓: 710 手
  实际杠杆: 0.06x

测试配置: 高持仓（10000手）
  最终权益: $1,435,553.72
  总收益率: 43.56%
  年化波动率: 53.07%
  最大回撤: -57.50%
  夏普比率: 0.82
  平均持仓: 7100 手
  实际杠杆: 0.54x

测试配置: 完全杠杆（50000手）


INFO:src.core.backtest:回测完成，共执行 292 笔交易
INFO:src.core.backtest:最终权益: $-144,886.11


  最终权益: $-144,886.11
  总收益率: -114.49%
  年化波动率: 274.86%
  最大回撤: -114.49%
  夏普比率: -0.42
  平均持仓: 6692 手
  实际杠杆: 5.09x

📊 配置对比总结
            配置  最大持仓          最终权益     总收益率(%)   年化波动率(%)     最大回撤(%)      夏普比率  峰值保证金占用(%)        平均持仓     平均杠杆
当前配置（持仓限制100手）   100  1.004337e+06    0.433724   0.473803   -0.638023  0.915408    0.081139   70.503638 0.005680
   中等持仓（1000手）  1000  1.043502e+06    4.350165   4.778407   -6.349110  0.910380    0.811084  709.555376 0.056565
   高持仓（10000手） 10000  1.435554e+06   43.555372  53.067173  -57.500243  0.820759    8.105244 7100.118027 0.535479
  完全杠杆（50000手） 50000 -1.448861e+05 -114.488611 274.856153 -114.486090 -0.416540   25.947400 6691.982201 5.094281

✓ 对比图已保存: outputs/charts/leverage_comparison.png

💡 结论和建议

观察结果:
  - 持仓限制越高，收益和波动率都越大
  - 杠杆充分利用后，波动率会显著提升
  - 需要在收益和风险之间找到平衡点

建议:
  1. 如果追求稳健：保持 100-1000 手
  2. 如果追求收益：使用 10000-50000 手（但风险极高）
  3. 最佳实践：使用动态持仓管理，根据市场波动率调整


# 🆕 回归任务信号生成功能测试

测试新增的回归任务阈值逻辑和滚动分位数功能。

**主要功能**：
1. **固定阈值模式**：使用预设的上下阈值（如±3%）
2. **滚动分位数模式**：使用动态计算的分位数阈值（推荐）
3. **信号维持**：配合 `signal_holding_days` 参数使用

**优势**：
- ✅ 回归任务也能生成 1, 0, -1 离散信号
- ✅ 滚动分位数自适应市场变化
- ✅ 避免固定阈值在不同市场环境下失效

In [ ]:
# 运行回归信号生成测试
print("运行回归信号生成测试...")
print("="*80)

# 方法1: 直接运行测试脚本
exec(open('examples/test_regression_signals.py', encoding='utf-8').read())

print("\n" + "="*80)
print("✅ 回归信号生成功能测试完成！")
print("="*80)